# SupportOps: the complete workflow

This notebook shows the final local support workflow one step at a time. It uses fictional orders and policies. The first sections run without a network call. The live Groq sections are optional.

Select the project's `.venv` Python kernel, restart it, and run cells from top to bottom.

Flow: question -> Groq agent -> allowed tool -> answer. Read-only answers come from Groq; cancellation outcomes use a fixed response from verified data. A cancellation pauses before any database update and needs human approval. LangSmith tracing is optional.

## 1. Load the project

This cell loads the settings, synthetic orders, and store policies. It also creates a temporary SQLite database for this notebook, so the examples do not change the Streamlit demo data.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from support_ops.agent import build_support_agent
from support_ops.config import find_project_root, load_settings, project_file
from support_ops.evaluation import evaluate_scenarios, load_scenarios
from support_ops.graph import build_order_graph
from support_ops.policies import load_policies, search_policies
from support_ops.storage import SupportStore
from support_ops.tools import load_orders

root = find_project_root()
settings = load_settings(root)
orders = load_orders(project_file(root, settings.files.orders))
policies = load_policies(project_file(root, "data/policies.json"))
temporary_data = TemporaryDirectory(prefix="supportops-notebook-")
store = SupportStore(Path(temporary_data.name))
store.initialize(project_file(root, settings.files.orders))
print({"model": settings.llm.model, "orders": len(orders), "policies": len(policies)})

## 2. Read a policy and an order

A tool result is data that the LLM can use in its answer. The policy tool returns a source ID. The order lookup returns only an order owned by the current demo customer. A missing order and another customer's order look the same.

In [ ]:
policy = search_policies(policies, "Quelle est la politique de retour ?")
own_order = store.get_order(settings.demo.customer_id, "ORD-1042")
foreign_order = store.get_order(settings.demo.customer_id, "ORD-2099")
assert policy["source_id"] == "POL-RETURNS"
assert own_order["found"] and not foreign_order["found"]
print(json.dumps({"policy": policy, "own_order": own_order, "other_customer": foreign_order}, ensure_ascii=False, indent=2))

## 3. See the agent graph

The agent node asks Groq what to do. Read tools return their results to the agent for an answer. Cancellation pauses inside execute_tools for approval, then produces a fixed answer from the verified status and goes to END. The loop is bounded to four tool calls.

The drawing below comes from the actual compiled graph. The following local example uses a scripted model so it makes no API call.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

class ScriptedCancelModel:
    def bind_tools(self, tools, tool_choice="auto"):
        return self

    def invoke(self, messages):
        if isinstance(messages[-1], ToolMessage):
            result = json.loads(messages[-1].content)
            return AIMessage(content=f"Demo result: {result.get('outcome')}.")
        return AIMessage(content="", tool_calls=[{
            "name": "request_cancel",
            "args": {"order_id": "ORD-1050"},
            "id": "cancel-demo",
        }])

scripted_graph = build_support_agent(
    root, settings.demo.customer_id, store=store,
    model=ScriptedCancelModel(), checkpointer=InMemorySaver()
)
view = scripted_graph.get_graph()
positions = {
    "__start__": (0.12, 0.72),
    "agent": (0.38, 0.72),
    "execute_tools": (0.68, 0.72),
    "__end__": (0.82, 0.20),
}
fig, axis = plt.subplots(figsize=(10, 5))
axis.set(xlim=(0, 1), ylim=(0, 1))
axis.axis("off")
arrows = [
    ((0.22, 0.72), (0.28, 0.72), "arc3"),
    ((0.48, 0.72), (0.58, 0.72), "arc3"),
    ((0.68, 0.78), (0.38, 0.78), "arc3,rad=0.50"),
    ((0.38, 0.66), (0.70, 0.20), "arc3,rad=0.18"),
    ((0.68, 0.66), (0.82, 0.26), "arc3,rad=-0.12"),
]
for start, end, curve in arrows:
    axis.annotate("", xy=end, xytext=start,
                  arrowprops={"arrowstyle": "->", "lw": 1.8, "connectionstyle": curve})
for node_id in view.nodes:
    x, y = positions[node_id]
    label = {"__start__": "START", "__end__": "END"}.get(node_id, node_id)
    box = FancyBboxPatch((x - 0.10, y - 0.05), 0.20, 0.10,
                         boxstyle="round,pad=0.01", facecolor="#dbeafe",
                         edgecolor="#334155")
    axis.add_patch(box)
    axis.text(x, y, label, ha="center", va="center")
axis.set_title("SupportOps agent and tool loop")
plt.show()

## 4. Watch approval and replay protection

The first run asks for cancellation and pauses. The order is still processing. Denying the request keeps it unchanged. A second run is approved: the status becomes cancelled. Repeating the same action ID returns the saved receipt. A new cancellation request must then say the order is already cancelled, without pausing again.

This is a simulated local action, not a real store operation.

In [ ]:
deny_config = {"configurable": {"thread_id": str(uuid4())}}
pending = scripted_graph.invoke({
    "messages": [HumanMessage(content="Annule ma commande ORD-1050.")],
    "tool_calls_used": 0,
}, config=deny_config)
assert "__interrupt__" in pending
assert store.get_order(settings.demo.customer_id, "ORD-1050")["status"] == "processing"
print("Paused for approval:", pending["__interrupt__"][0].value)
scripted_graph.invoke(Command(resume=False), config=deny_config)
denied_status = store.get_order(settings.demo.customer_id, "ORD-1050")["status"]

approve_config = {"configurable": {"thread_id": str(uuid4())}}
pending = scripted_graph.invoke({
    "messages": [HumanMessage(content="Annule ma commande ORD-1050.")],
    "tool_calls_used": 0,
}, config=approve_config)
assert "__interrupt__" in pending
scripted_graph.invoke(Command(resume=True), config=approve_config)
approved_status = store.get_order(settings.demo.customer_id, "ORD-1050")["status"]
replay = store.cancel_order(settings.demo.customer_id, "ORD-1050", "CUST-001:cancel-demo")
assert denied_status == "processing" and approved_status == "cancelled" and replay["replayed"]
repeat_config = {"configurable": {"thread_id": str(uuid4())}}
repeat = scripted_graph.invoke({
    "messages": [HumanMessage(content="Annule ma commande ORD-1050.")],
    "tool_calls_used": 0,
}, config=repeat_config)
assert "__interrupt__" not in repeat
assert repeat["last_tool_result"]["status"] == "cancelled"
assert "déjà annulée" in repeat["messages"][-1].content
print({"after_denial": denied_status, "after_approval": approved_status, "same_request_replayed": replay["replayed"]})
print("Repeated request:", repeat["messages"][-1].content)

## 5. Check the routing baseline with DeepEval

Each reviewed question has an expected category, path, and answer. The exact-match DeepEval metric checks the answer. The local evaluator also checks the graph path. These offline checks do not measure Groq quality.

In [ ]:
from deepeval.metrics import ExactMatchMetric
from deepeval.test_case import LLMTestCase

baseline = build_order_graph(orders, settings.demo.customer_id)
scenarios = load_scenarios(project_file(root, "data/scenarios.json"))
rows = evaluate_scenarios(baseline, scenarios)
for scenario, row in zip(scenarios, rows, strict=True):
    metric = ExactMatchMetric()
    metric.measure(LLMTestCase(
        input=scenario.question,
        actual_output=row["answer"],
        expected_output=scenario.answer,
    ))
    assert row["passed"] and metric.is_successful(), scenario.id
print(f"DeepEval exact answer and path checks: {len(rows)}/{len(rows)} passed.")

## 6. Try the real Groq agent

Set RUN_LIVE_AGENT to True only when GROQ_API_KEY is in the local .env file. A live run consumes Groq quota. The next cell asks about an order and a return policy. It prints selected tool names and the final answers.

To send traces to LangSmith, add LANGSMITH_API_KEY and LANGSMITH_TRACING=true to .env, then restart this kernel. Never put keys in a notebook cell.

In [ ]:
RUN_LIVE_AGENT = False

In [ ]:
if RUN_LIVE_AGENT:
    live_store = SupportStore(Path(temporary_data.name) / "live")
    live_store.initialize(project_file(root, settings.files.orders))
    live_graph = build_support_agent(
        root, settings.demo.customer_id, store=live_store,
        checkpointer=InMemorySaver()
    )
    for question in (
        "Quel est le statut de ma commande ORD-1042 ?",
        "Quelle est la politique de retour ?",
    ):
        state = live_graph.invoke(
            {"messages": [HumanMessage(content=question)], "tool_calls_used": 0},
            config={"configurable": {"thread_id": str(uuid4())}},
        )
        tools = [
            call["name"] for message in state["messages"]
            if isinstance(message, AIMessage) for call in message.tool_calls
        ]
        answer = next(
            str(message.content) for message in reversed(state["messages"])
            if isinstance(message, AIMessage) and not message.tool_calls
        )
        print(json.dumps({"question": question, "tools": tools, "answer": answer}, ensure_ascii=False, indent=2))
else:
    print("Live Groq calls skipped. Set RUN_LIVE_AGENT = True to run them.")

## 7. Pause a real LLM cancellation

With live mode on, this cell asks Groq to cancel an eligible fictional order. It should stop at the approval request while the status is still processing. The next cell resumes only if you change APPROVE_LIVE_CANCELLATION to True yourself.

In [ ]:
live_pending = None
if RUN_LIVE_AGENT:
    live_cancel_config = {"configurable": {"thread_id": str(uuid4())}}
    live_pending = live_graph.invoke(
        {"messages": [HumanMessage(content="Annule ma commande ORD-1050.")], "tool_calls_used": 0},
        config=live_cancel_config,
    )
    print({"paused": "__interrupt__" in live_pending,
           "order_status": live_store.get_order(settings.demo.customer_id, "ORD-1050")["status"]})
    if "__interrupt__" in live_pending:
        print(live_pending["__interrupt__"][0].value)
else:
    print("Live cancellation skipped.")

In [ ]:
APPROVE_LIVE_CANCELLATION = False
if RUN_LIVE_AGENT and live_pending is not None and "__interrupt__" in live_pending:
    if APPROVE_LIVE_CANCELLATION:
        finished = live_graph.invoke(Command(resume=True), config=live_cancel_config)
        print({"status": live_store.get_order(settings.demo.customer_id, "ORD-1050")["status"],
               "answer": str(finished["messages"][-1].content)})
    else:
        print("Waiting for your decision. The order was not changed.")

## Use the full app

Run this from the project root: ./.venv/Scripts/python.exe -m streamlit run app.py

The app keeps conversations in a local SQLite checkpoint, shows demo orders, and gives Approve and Deny buttons when cancellation pauses. The separate script scripts/run_evals.py runs offline DeepEval checks and optional live Groq and judge checks. See README.md for the exact commands and limitations.